In [1]:
import pandas as pd
import numpy as np

from sksurv.ensemble import RandomSurvivalForest
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sksurv.util import Surv

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sksurv.linear_model import CoxnetSurvivalAnalysis


In [2]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    nacc_data_csv = "/content/drive/MyDrive/bachelor/nacc_data_2025.csv"
else:
    nacc_data_csv = "nacc_data_2025.csv"

In [3]:
nacc_raw = pd.read_csv(nacc_data_csv, delimiter='\t')

/var/folders/g9/j9c8b7vs3m3frqcxprr6hf900000gn/T/ipykernel_42639/2964923166.py:1: DtypeWarning: Columns (8,10,12,14,25,29,31,40,42,44,46,48,50,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,142,194,197,199,205,207,209,211,213,215,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,374,376,378,396,398,409,422,429,469,549,572,580,605,640,673,676,693,704,710,763,765,766,767,768,774,797,809,810,818,819,820,821,831,853,856,859) have mixed types. Specify dtype option on import or set low_memory=False.
  nacc_raw = pd.read_csv(nacc_data_csv, delimiter='\t')


# Helper functions

These function were created and implemented suring EDA phase, and they will be used for the further preprocessing

In [4]:
def filter_columns_by_missing_pattern(df, reference_col='HIV'):
    if reference_col not in df.columns:
        raise KeyError(f"Column '{reference_col}' not found in dataframe.")

    reference_mask = df[reference_col].isna().to_numpy()

    forward_columns = []
    opposite_columns = []

    for col in df.columns:
        col_mask = df[col].isna().to_numpy()

        if np.array_equal(col_mask, reference_mask):
            forward_columns.append(col)
        elif np.array_equal(col_mask, ~reference_mask):
            opposite_columns.append(col)

    kept_columns = forward_columns + opposite_columns
    dropped_columns = [col for col in df.columns if col not in kept_columns]
    filtered_df = df[kept_columns].copy()

    return filtered_df, forward_columns, opposite_columns, dropped_columns

In [5]:
LOW_MISSINGNESS_THRESHOLD = 20
HIGH_MISSINGNESS_THRESHOLD = 80

In [6]:
def define_missingnes(df):
  missing_percantage_per_column = df.isna().sum() / len(df) * 100

  low_missing = missing_percantage_per_column[missing_percantage_per_column < LOW_MISSINGNESS_THRESHOLD].index.tolist()
  medium_missing = missing_percantage_per_column[(missing_percantage_per_column >= LOW_MISSINGNESS_THRESHOLD) & (missing_percantage_per_column < HIGH_MISSINGNESS_THRESHOLD)].index.tolist()
  high_missing = missing_percantage_per_column[missing_percantage_per_column >= HIGH_MISSINGNESS_THRESHOLD].index.tolist()

  return low_missing, medium_missing, high_missing

In [7]:
NOT_COLLECTED_PLACEHOLDER_VALUE = -99.0
important_very_imbalanced_columns = ["NACCFADM", "ELAT", "GAMES", "MOGAIT", "MOSLOW", "BRNINJ", "OTHPSY"]

In [8]:
def define_categorical_and_continuous_columns(df):
    result_df = df.copy()
    unusual_categorical = ['NACCBEHF']
    categorical_cols = []
    continuous_cols = []
    categorical_cols_by_unique_count = {}

    for col in result_df.columns:
        if col == 'EVENT_MCI':
            continue
        
        n_unique = result_df[col].nunique(dropna=True)
        if n_unique == 1:
            result_df.drop(columns=[col], inplace=True)
            continue
        if 2 <= n_unique <= 10 or col in unusual_categorical:
            categorical_cols.append(col)
            if n_unique not in categorical_cols_by_unique_count:
                categorical_cols_by_unique_count[n_unique] = []
            categorical_cols_by_unique_count[n_unique].append(col)
            continue
        continuous_cols.append(col)

    return categorical_cols, continuous_cols, categorical_cols_by_unique_count

In [ ]:
def clean_columns(df):
    df_clean = df.copy()
    all_categorical_cols, all_continuous_cols, categorical_cols_by_unique_count = define_categorical_and_continuous_columns(df_clean)

    for col in all_categorical_cols:
        if col not in df_clean.columns:
            continue
        col_value_proportions = df_clean[col].value_counts(normalize=True, dropna=True)
        if col_value_proportions.iloc[0] > 0.99 and col not in important_very_imbalanced_columns:
            print(f"Column '{col}' has very high imbalance ({col_value_proportions.iloc[0]:.2%} of one category); consider to drop it.")
            df_clean.drop(columns=[col], inplace=True)

    for col in all_continuous_cols:
        if col not in df_clean.columns:
            continue
        col_var = df_clean[col].var(numeric_only=True)
        if col_var == 0.0000:
            df_clean.drop(columns=[col], inplace=True)
            continue
        if col_var < 0.01 and col not in important_very_imbalanced_columns:
            print(f"Column '{col}' has very low variance ({col_var:.6f}); consider to drop it.")
            df_clean.drop(columns=[col], inplace=True)

    # keep column lists synchronized with the actually retained dataframe
    filtered_categorical_cols = [col for col in all_categorical_cols if col in df_clean.columns]
    filtered_continuous_cols = [col for col in all_continuous_cols if col in df_clean.columns]

    return df_clean, filtered_categorical_cols, filtered_continuous_cols


In [10]:
class RareCategoryCollapser(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01, categorical_cols=None, non_collected_placeholder=None):
        self.threshold = threshold
        self.categorical_cols = categorical_cols
        self.non_collected_placeholder = non_collected_placeholder
        self.rare_categories_ = {}

    def fit(self, X, y=None):
        if self.non_collected_placeholder is None:
            raise ValueError('non_collected_placeholder cannot be None')

        df_result = X.copy()
        self.feature_names_in_ = np.asarray(df_result.columns, dtype=object)
        for col in self.categorical_cols:
            if col not in df_result.columns:
                continue

            value_counts = df_result[col][df_result[col] != self.non_collected_placeholder].value_counts(normalize=True)
            rare_categories = value_counts[value_counts < self.threshold].index.tolist()
            self.rare_categories_[col] = rare_categories

        return self

    def transform(self, X):
        df_result = X.copy()

        for col, rare_categories in self.rare_categories_.items():
            if col in df_result.columns and len(rare_categories) > 0:
                rare_categories_str = ', '.join(map(str, rare_categories))
                df_result[col] = df_result[col].replace(rare_categories, f'{col}_{rare_categories_str}')
        return df_result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features, dtype=object)


In [11]:
class CustomOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, categorical_cols=None, selected_features_subset=None):
        self.categorical_cols = categorical_cols
        self.selected_features_subset = selected_features_subset
        self.encoder = None

    def fit(self, X, y=None):
        df_result = X.copy()
        categorical_df = df_result[self.categorical_cols].astype(str)

        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.encoder.fit(categorical_df)

        feature_names = self.encoder.get_feature_names_out(self.categorical_cols)
        self.keep_columns_indices_ = [
            i for i, name in enumerate(feature_names)
            if not name.endswith(f'_{NOT_COLLECTED_PLACEHOLDER_VALUE}') and not name.endswith(f'_{NOT_COLLECTED_PLACEHOLDER_VALUE:.1f}')
        ]
        self.feature_names_out_ = feature_names[self.keep_columns_indices_]

        return self

    def transform(self, X):
        df_result = X.copy()
        categorical_df = df_result[self.categorical_cols].astype(str)
        encoded_array = self.encoder.transform(categorical_df)

        encoded_array = encoded_array[:, self.keep_columns_indices_]
        return encoded_array

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_out_, dtype=object)


In [12]:
class CustomConstantImputer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=None):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        if self.fill_value is None:
            raise ValueError('fill_value cannot be None')
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        df_result = X.copy()
        df_result.fillna(self.fill_value, inplace=True)
        return df_result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features, dtype=object)


In [13]:
def decode_preprocessed_feature_name(feature_name, categorical_cols, continuous_cols):
    if feature_name.startswith('categorical__'):
        remainder = feature_name.replace('categorical__', '', 1)
        matching = [
            col for col in categorical_cols
            if remainder == col or remainder.startswith(f'{col}_')
        ]
        if len(matching) > 0:
            return max(matching, key=len)
        return remainder

    if feature_name.startswith('continuous__'):
        return feature_name.replace('continuous__', '', 1)

    return feature_name

# Structural dataset cleanup

This cleanup make overall dataset cleanup before applying splitting and preprocesiing pipeline

In [14]:
def subset_structural_cleanup(df, selected_features, model_categorical_cols, model_continuous_cols):
    result_df = df.copy()
    if 'HIV' in result_df.columns:
        result_df['group_missing_indicator'] = np.where(
            result_df['HIV'].isna(), 1, 0
        )

    raw_features = []
    for feature in selected_features:
        base_name = decode_preprocessed_feature_name(feature, model_categorical_cols, model_continuous_cols)

        if base_name in result_df.columns and base_name not in raw_features:
            raw_features.append(base_name)

    for required_col in ['TIME', 'EVENT_MCI']:
        if required_col in result_df.columns and required_col not in raw_features:
            raw_features.append(required_col)

    nacc_selected_features_df = result_df[raw_features].copy()

    low_missing, _, _ = define_missingnes(nacc_selected_features_df)
    if len(low_missing) > 0:
        nacc_selected_features_df.dropna(subset=low_missing, inplace=True)

    categorical_cols, continuous_cols, _ = define_categorical_and_continuous_columns(nacc_selected_features_df)

    return nacc_selected_features_df, categorical_cols, continuous_cols


In [15]:
def structural_cleanup(df):
  df = df.copy()

  # drop columns that are target leakage features
  # these columns themselves represent the MCI or do not hold relevant information for prediction
  print(f"Dropping useless columns and columns represented the MCI diagnosis")
  df.drop(columns=['NACCACTV', 'NACCADMD', 'NACCALZD', 'NACCALZP', 'PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF'], inplace=True, errors='ignore')
  df.drop(columns=['NACCMCII', 'NACCNORM', 'COGSTAT', 'VISITDAY', 'VISITYR', 'VISITMO', 'NACCETPR'], inplace=True, errors='ignore')
  df.drop(columns=['NACCID'], inplace=True, errors='ignore')

  # define column missingness
  print(f"Defining missingness")
  low_missing, medium_missing, high_missing = define_missingnes(df)

  # clean missingness
  print(f"Filtering columns by missing pattern")
  nacc_pattern_filtered, forward_cols, opposite_cols, pattern_dropped_cols = filter_columns_by_missing_pattern(
      df[medium_missing]
  )

  columns_to_proceed = nacc_pattern_filtered.columns.tolist() + low_missing
  nacc_missing_pattern_filtered = df[columns_to_proceed].copy()

  print(f"Complete-case analysis on low-missing columns")
  low_missing_in_filtered = [c for c in low_missing if c in nacc_missing_pattern_filtered.columns]
  if len(low_missing_in_filtered) > 0:
    nacc_missing_free_v2_v3 = nacc_missing_pattern_filtered.dropna(
      subset=low_missing_in_filtered
    ).copy()
  else:
    nacc_missing_free_v2_v3 = nacc_missing_pattern_filtered.copy()

  print(f"Creating missingness indicator")

  nacc_missing_free_v2_v3['group_missing_indicator'] = np.where(
      nacc_missing_free_v2_v3['HIV'].isna(), 1, 0
  )

  # clean columns
  print(f"Cleaning columns")
  nacc_clean, nacc_categorical_columns, continuous_columns = clean_columns(nacc_missing_free_v2_v3)

  return nacc_clean, nacc_categorical_columns, continuous_columns


# Preprocessing pipeline

In [16]:
def build_preprocessing_pipeline(categorical_columns, continuous_columns, selected_features_subset=None):
  # Categorical pipeline
  categorical_pipeline = Pipeline([
    ('imputer', CustomConstantImputer(fill_value=NOT_COLLECTED_PLACEHOLDER_VALUE)),
    ('rare_collapser', RareCategoryCollapser(
      threshold=0.01,
      categorical_cols=categorical_columns,
      non_collected_placeholder=NOT_COLLECTED_PLACEHOLDER_VALUE,
    )),
    ('encoder', CustomOneHotEncoder(categorical_cols=categorical_columns, selected_features_subset=selected_features_subset)),
  ]).set_output(transform='pandas')

  # Continuous features pipeline
  continious_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
  ]).set_output(transform='pandas')

  columns_preprocessing_pipeline = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_columns),
    ('continuous', continious_pipeline, continuous_columns),
  ]).set_output(transform='pandas')

  return Pipeline([
    ('columns_preprocessing', columns_preprocessing_pipeline),
  ]).set_output(transform='pandas')


create feature selection up to 95 features with rsf vimp

Skontrolovat podmienky na feature selection. Nutne podmienky musia byt splnene, dostatocne podmienky moszu ale nemusia, staci aj bez nich, ale idealne c nimi.

Overit, ci ako feature selection pracuje c kategorickyi premennami.

NACCBEHF, 

# Feature selection

Accordingto rule of thumb, that each feature should contain at least 10 records, we select 95 features to proceed further

In [17]:
# we definetelly need missing indicator column, so we will select top 94 features
N_IMPORTANT_FEATURES = 94

In [18]:
def rsf_vimp_feature_selection(
    x_data,
    y_data,
    n_features=N_IMPORTANT_FEATURES,
    n_estimators=200,
    n_repeats=8,
    random_state=42,
):
    # rsf = RandomSurvivalForest(
    #     n_estimators=n_estimators,
    #     random_state=random_state,
    #     max_features='sqrt',
    # )
    # rsf.fit(x_data, y_data)

    # importance = permutation_importance(
    #     rsf,
    #     x_data,
    #     y_data,
    #     n_repeats=n_repeats,
    #     random_state=random_state,
    # )

    # feature_importance = pd.DataFrame(
    #     {
    #         'importances_mean': importance['importances_mean'],
    #         'importances_std': importance['importances_std'],
    #     },
    #     index=x_data.columns,
    # )
    # feature_importance['importances_mean_abs'] = np.abs(feature_importance['importances_mean'])
    # feature_importance = feature_importance.sort_values(by='importances_mean_abs', ascending=False)

    # selected_features = feature_importance.head(n_features).index.tolist()

    cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=0.01, alpha_min_ratio=0.01)
    cox_lasso.fit(x_data, y_data)

    coefficients_lasso = pd.DataFrame(cox_lasso.coef_, index=x_data.columns, columns=np.round(cox_lasso.alphas_, 5))

    optimal_alpha_i = -1
    feature_importance = pd.DataFrame({
        # "feature": x_data.columns,
        "importance": coefficients_lasso.iloc[:, optimal_alpha_i],
        "importance_abs": np.abs(coefficients_lasso.iloc[:, optimal_alpha_i])
    })

    selected_features = feature_importance.sort_values(by='importance_abs', ascending=False).head(n_features).index.tolist()

    return selected_features, feature_importance


# Dataset preprocessing - 1st stage

## Raw dataset structural cleaning

In [19]:
nacc_clean, categorical_cols, continuous_cols = structural_cleanup(nacc_raw)

model_categorical_cols = [c for c in categorical_cols if c not in ['EVENT_MCI', 'TIME']]
model_continuous_cols = [c for c in continuous_cols if c not in ['EVENT_MCI', 'TIME']]

Dropping useless columns and columns represented the MCI diagnosis
Defining missingness
Filtering columns by missing pattern
Complete-case analysis on low-missing columns
Creating missingness indicator
Cleaning columns
Column 'PDOTHR' has very high imbalance (99.79% of one category); consider to drop it.
Column 'COMMUN' has very high imbalance (99.12% of one category); consider to drop it.
Column 'PERSCARE' has very high imbalance (99.87% of one category); consider to drop it.
Column 'DEL' has very high imbalance (99.62% of one category); consider to drop it.
Column 'HALL' has very high imbalance (99.82% of one category); consider to drop it.
Column 'COGVIS' has very high imbalance (99.39% of one category); consider to drop it.
Column 'COGOTHR' has very high imbalance (99.96% of one category); consider to drop it.
Column 'BEVHALL' has very high imbalance (99.87% of one category); consider to drop it.
Column 'BEAHALL' has very high imbalance (99.92% of one category); consider to drop it

## Split to train test

In [20]:
train_df, test_df = train_test_split(
    nacc_clean,
    test_size=0.2,
    random_state=42,
    stratify=nacc_clean['EVENT_MCI'],
)

X_train_full = train_df.drop(columns=['TIME', 'EVENT_MCI'])
X_test_full = test_df.drop(columns=['TIME', 'EVENT_MCI'])
y_train = Surv.from_dataframe('EVENT_MCI', 'TIME', train_df)
y_test = Surv.from_dataframe('EVENT_MCI', 'TIME', test_df)

## Preprocess dataset

In [31]:
preprocessing_pipeline = build_preprocessing_pipeline(model_categorical_cols, model_continuous_cols)

X_train_preprocessed = preprocessing_pipeline.fit_transform(X_train_full)
X_test_preprocessed = preprocessing_pipeline.transform(X_test_full)

X_train_preprocessed.shape, X_test_preprocessed.shape

X_train_preprocessed


,categorical__DECCLCOG_0.0,categorical__DECCLCOG_1.0,categorical__DECCLBE_0.0,categorical__DECCLBE_1.0,categorical__DECCLMOT_0.0,categorical__DECCLMOT_1.0,categorical__LBDEVAL_0.0,categorical__LBDEVAL_1.0,categorical__FTLDEVAL_0.0,categorical__FTLDEVAL_1.0,...,continuous__BPDIAS,continuous__HRATE,continuous__NACCGDS,continuous__ANIMALS,continuous__VEG,continuous__TRAILA,continuous__TRAILB,continuous__NACCAGEB,continuous__NACCAMD,continuous__NACCBMI
16206,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.248442,1.002464,-0.643759,-0.419435,-0.287334,-0.094423,0.230359,-1.168180,-0.400928,-0.415107
3790,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.150736,-0.424803,0.509434,1.196181,1.131709,-1.389660,-1.025812,-4.282740,-1.130504,-0.941757
2725,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.924028,2.429731,2.239222,-0.060410,-1.233362,-0.822994,-0.470198,0.740744,3.003756,0.765317
15523,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.736971,0.717011,1.086030,1.016668,0.185680,-0.742041,0.496087,-0.163483,-0.157737,-1.105200
12068,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.150736,0.336406,0.509434,-0.060410,-0.996855,1.119862,0.592716,1.444031,-0.644120,-0.923597
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8152,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.443853,1.573371,0.509434,-0.419435,-0.996855,1.362719,0.882601,0.037456,-0.400928,2.345268
2780,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.127794,-0.329652,-0.067162,0.119103,-0.287334,0.472243,0.109573,0.941683,0.571839,0.565553
142,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-0.240087,0.907313,1.086030,1.196181,-1.469869,-0.580137,-0.663455,-3.880861,-0.644120,-1.777134
1744,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-0.240087,-0.615105,-0.643759,0.119103,-1.233362,-0.013471,-0.131998,0.037456,-0.644120,-0.669352


Restore the initial columns names, in order to repeat the process of cleaning and preprocessing on the subset of selected features

In [32]:
preprocessed_feature_names = preprocessing_pipeline.named_steps['columns_preprocessing'].get_feature_names_out()

X_train_preprocessed_df = pd.DataFrame(
    X_train_preprocessed,
    index=X_train_full.index,
    columns=preprocessed_feature_names,
)
X_test_preprocessed_df = pd.DataFrame(
    X_test_preprocessed,
    index=X_test_full.index,
    columns=preprocessed_feature_names,
)

X_train_preprocessed_df

,categorical__DECCLCOG_0.0,categorical__DECCLCOG_1.0,categorical__DECCLBE_0.0,categorical__DECCLBE_1.0,categorical__DECCLMOT_0.0,categorical__DECCLMOT_1.0,categorical__LBDEVAL_0.0,categorical__LBDEVAL_1.0,categorical__FTLDEVAL_0.0,categorical__FTLDEVAL_1.0,...,continuous__BPDIAS,continuous__HRATE,continuous__NACCGDS,continuous__ANIMALS,continuous__VEG,continuous__TRAILA,continuous__TRAILB,continuous__NACCAGEB,continuous__NACCAMD,continuous__NACCBMI
16206,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.248442,1.002464,-0.643759,-0.419435,-0.287334,-0.094423,0.230359,-1.168180,-0.400928,-0.415107
3790,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.150736,-0.424803,0.509434,1.196181,1.131709,-1.389660,-1.025812,-4.282740,-1.130504,-0.941757
2725,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.924028,2.429731,2.239222,-0.060410,-1.233362,-0.822994,-0.470198,0.740744,3.003756,0.765317
15523,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.736971,0.717011,1.086030,1.016668,0.185680,-0.742041,0.496087,-0.163483,-0.157737,-1.105200
12068,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.150736,0.336406,0.509434,-0.060410,-0.996855,1.119862,0.592716,1.444031,-0.644120,-0.923597
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8152,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.443853,1.573371,0.509434,-0.419435,-0.996855,1.362719,0.882601,0.037456,-0.400928,2.345268
2780,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.127794,-0.329652,-0.067162,0.119103,-0.287334,0.472243,0.109573,0.941683,0.571839,0.565553
142,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-0.240087,0.907313,1.086030,1.196181,-1.469869,-0.580137,-0.663455,-3.880861,-0.644120,-1.777134
1744,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,...,-0.240087,-0.615105,-0.643759,0.119103,-1.233362,-0.013471,-0.131998,0.037456,-0.644120,-0.669352


## Selecting features

In [23]:
selected_features, rsf_vimp_importance = rsf_vimp_feature_selection(
    X_train_preprocessed_df,
    y_train,
    n_features=N_IMPORTANT_FEATURES,
)

X_train = X_train_preprocessed_df[selected_features]
X_test = X_test_preprocessed_df[selected_features]

print(f'Selected {len(selected_features)} preprocessed features')
X_train.shape, X_test.shape


Selected 94 preprocessed features


((5649, 94), (1413, 94))

In [24]:
selected_features

['continuous__NACCAGEB',
 'continuous__VEG',
 'categorical__NACCNOVS_0',
 'categorical__NACCNOVS_1',
 'continuous__TRAILB',
 'continuous__ANIMALS',
 'categorical__DECSUB_0.0',
 'categorical__DECSUB_1.0',
 'continuous__SMOKYRS',
 'categorical__IRR_0.0',
 'categorical__IRR_1.0',
 'categorical__CDRSUM_0.0',
 'continuous__NACCGDS',
 'categorical__WHODIDDX_1.0',
 'categorical__DECIN_0.0',
 'categorical__DECIN_1.0',
 'categorical__MEMPROB_0.0',
 'categorical__MEMPROB_1.0',
 'categorical__MEMORY_0.0',
 'categorical__NACCREFR_1.0',
 'categorical__CDRGLOB_0.5',
 'categorical__CDRGLOB_0.0',
 'categorical__NACCREFR_8.0',
 'categorical__MEMORY_0.5',
 'categorical__NACCAC_0.0',
 'categorical__NACCAC_1.0',
 'categorical__WHODIDDX_2.0',
 'categorical__CDRSUM_0.5',
 'categorical__DEPOTHR_1.0',
 'categorical__DEPOTHR_0.0',
 'continuous__TRAILA',
 'categorical__ENERGY_0.0',
 'categorical__ENERGY_1.0',
 'categorical__TRAUMBRF_0.0',
 'continuous__NACCAMD',
 'categorical__HEARAID_0.0',
 'categorical__HEARA

# Dataset preprocessing - 2nd stage

Repeat the entire preprocessing on the subset of selected features.\
When we made complete-case analysis on the low-missingness values there was remained only 7000 records, which can be considered as enough number of records. Although, we want torepeat the entire process but on the subset of selected features, that way it can remain more records in the final processed dataset.

## Cleanup

In [25]:
nacc_selected_features_clean, selected_categorical_cols, selected_continuous_cols = subset_structural_cleanup(
    nacc_raw,
    selected_features,
    model_categorical_cols,
    model_continuous_cols
)

nacc_selected_features_clean.shape

(10873, 62)

## Splitting

In [26]:
selected_train_df, selected_test_df = train_test_split(
    nacc_selected_features_clean,
    test_size=0.2,
    random_state=42,
    stratify=nacc_selected_features_clean['EVENT_MCI'],
)

selected_X_train_full = selected_train_df.drop(columns=['TIME', 'EVENT_MCI'])
selected_X_test_full = selected_test_df.drop(columns=['TIME', 'EVENT_MCI'])
selected_y_train = Surv.from_dataframe('EVENT_MCI', 'TIME', selected_train_df)
selected_y_test = Surv.from_dataframe('EVENT_MCI', 'TIME', selected_test_df)

selected_model_categorical_cols = [c for c in selected_categorical_cols if c not in ['EVENT_MCI', 'TIME']]
selected_model_continuous_cols = [c for c in selected_continuous_cols if c not in ['EVENT_MCI', 'TIME']]

## Preprocessing

In [27]:
selected_preprocessing_pipeline = build_preprocessing_pipeline(
    selected_model_categorical_cols,
    selected_model_continuous_cols,
    selected_features
)

selected_X_train_processed = selected_preprocessing_pipeline.fit_transform(selected_X_train_full)
selected_X_test_processed = selected_preprocessing_pipeline.transform(selected_X_test_full)

print(f'Selected-clean train shape: {selected_X_train_processed.shape}')
print(f'Selected-clean test shape: {selected_X_test_processed.shape}')

Selected-clean train shape: (8698, 140)
Selected-clean test shape: (2175, 140)


In [28]:
selected_X_train_processed.columns.tolist()

['categorical__NACCNOVS_0',
 'categorical__NACCNOVS_1',
 'categorical__DECSUB_0.0',
 'categorical__DECSUB_1.0',
 'categorical__IRR_0.0',
 'categorical__IRR_1.0',
 'categorical__WHODIDDX_1.0',
 'categorical__WHODIDDX_2.0',
 'categorical__DECIN_0.0',
 'categorical__DECIN_1.0',
 'categorical__MEMPROB_0.0',
 'categorical__MEMPROB_1.0',
 'categorical__MEMORY_0.0',
 'categorical__MEMORY_0.5',
 'categorical__MEMORY_MEMORY_1.0',
 'categorical__NACCREFR_1.0',
 'categorical__NACCREFR_2.0',
 'categorical__NACCREFR_8.0',
 'categorical__CDRGLOB_0.0',
 'categorical__CDRGLOB_0.5',
 'categorical__CDRGLOB_CDRGLOB_1.0',
 'categorical__NACCAC_0.0',
 'categorical__NACCAC_1.0',
 'categorical__DEPOTHR_0.0',
 'categorical__DEPOTHR_1.0',
 'categorical__ENERGY_0.0',
 'categorical__ENERGY_1.0',
 'categorical__TRAUMBRF_0.0',
 'categorical__TRAUMBRF_2.0',
 'categorical__TRAUMBRF_TRAUMBRF_1.0',
 'categorical__HEARAID_0.0',
 'categorical__HEARAID_1.0',
 'categorical__NPIQINF_1.0',
 'categorical__NPIQINF_2.0',
 'cat

In [29]:
selected_X_train_processed.shape

(8698, 140)

# Store datasets into csv files

We store the finally preprocessed datasets into csv file for further model training and evaluation